In [26]:
# from langchain_core.documents import Document
# from langchain_community.document_loaders import PyMuPDFLoader

In [27]:
# document = Document(
#     page_content="",
#     metadata={
#         "source": "",
#         "pages": 18,
#     }
# )
# document

In [28]:
# loader = PyMuPDFLoader(
#     file_path='../data/(Minor Project) PatchFool.pdf',
#     mode='single', #'single', 'page',
# )
# document = loader.load()
# print(document)

# Data Ingestion and Vector DB

In [29]:
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter

In [30]:
def process_all_PDFs(dir_path):
    docs = []
    pdf_dir = Path(dir_path)
    pdf_files = list(pdf_dir.glob('**/*.pdf'))

    print(f'Found {len(pdf_files)} PDF files')

    for pdf_file in pdf_files:
        print(f"Processing -> {pdf_file.name} ...")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            document = loader.load()
            for page in document:
                page.metadata['source_file'] = pdf_file.name
                page.metadata['file_type'] = 'pdf'

            docs.extend(document)
            print(f'Loaded {document} pages')       
        except Exception as e:
            print(f'Error {e}')
    print(f'Total documents loaded: {len(docs)}')
    return docs

In [31]:
folder_path='../data/'
all_pdfs = process_all_PDFs(folder_path)

Found 1 PDF files
Processing -> (Minor Project) PatchFool.pdf ...
Loaded [Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-01-07T01:13:00+00:00', 'source': '..\\data\\(Minor Project) PatchFool.pdf', 'file_path': '..\\data\\(Minor Project) PatchFool.pdf', 'total_pages': 18, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-01-07T01:13:00+00:00', 'trapped': '', 'modDate': 'D:20250107011300Z', 'creationDate': 'D:20250107011300Z', 'page': 0, 'source_file': '(Minor Project) PatchFool.pdf', 'file_type': 'pdf'}, page_content='Published as a conference paper at ICLR 2022\nPATCH-FOOL: ARE VISION TRANSFORMERS ALWAYS\nROBUST AGAINST ADVERSARIAL PERTURBATIONS?\nYonggan Fu∗, Shunyao Zhang∗, Shang Wu∗, Cheng Wan & Yingyan Lin\nDepartment of Electrical and Computer Engineering, Rice University\n{yf22, sz74, sw99, chwan, yingyan.lin}@rice.edu\nABSTRACT\nVision transformers (ViTs) have recently set off

In [32]:
all_pdfs

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-01-07T01:13:00+00:00', 'source': '..\\data\\(Minor Project) PatchFool.pdf', 'file_path': '..\\data\\(Minor Project) PatchFool.pdf', 'total_pages': 18, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-01-07T01:13:00+00:00', 'trapped': '', 'modDate': 'D:20250107011300Z', 'creationDate': 'D:20250107011300Z', 'page': 0, 'source_file': '(Minor Project) PatchFool.pdf', 'file_type': 'pdf'}, page_content='Published as a conference paper at ICLR 2022\nPATCH-FOOL: ARE VISION TRANSFORMERS ALWAYS\nROBUST AGAINST ADVERSARIAL PERTURBATIONS?\nYonggan Fu∗, Shunyao Zhang∗, Shang Wu∗, Cheng Wan & Yingyan Lin\nDepartment of Electrical and Computer Engineering, Rice University\n{yf22, sz74, sw99, chwan, yingyan.lin}@rice.edu\nABSTRACT\nVision transformers (ViTs) have recently set off a new wave in neural architec-\nture design thanks to their record-break

# Chunking 

In [33]:
def split_docs(docs, chunk_size=500, chunk_overlap=200):
    enriched_docs = []
    for doc in docs:
        meta = doc.metadata
        content = doc.page_content

        # Try to enrich with author/title if available
        title = meta.get("title", "")
        authors = meta.get("authors", "")
        if title or authors:
            enriched_text = f"Title: {title}\nAuthors: {authors}\n\n{content}"
        else:
            enriched_text = content

        doc.page_content = enriched_text
        enriched_docs.append(doc)

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=['\n\n', ' ', '\n', '']
    )

    all_chunks = text_splitter.split_documents(enriched_docs)
    print(f"Formed {len(all_chunks)} chunks from {len(docs)} docs")
    return all_chunks

In [34]:
split_documents = split_docs(all_pdfs)

Formed 204 chunks from 18 docs


In [35]:
split_documents[0].page_content

'Published as a conference paper at ICLR 2022\nPATCH-FOOL: ARE VISION TRANSFORMERS ALWAYS\nROBUST AGAINST ADVERSARIAL PERTURBATIONS?\nYonggan Fu∗, Shunyao Zhang∗, Shang Wu∗, Cheng Wan & Yingyan Lin\nDepartment of Electrical and Computer Engineering, Rice University\n{yf22, sz74, sw99, chwan, yingyan.lin}@rice.edu\nABSTRACT\nVision transformers (ViTs) have recently set off a new wave in neural architec-\nture design thanks to their record-breaking performance in various vision tasks.\nIn parallel, to'

# Embeddings and Vector DB

In [36]:
import os
import uuid
import chromadb
import numpy as np

from sentence_transformers import SentenceTransformer
from chromadb.config import Settings
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Dict, Any, Tuple

In [37]:
class EmbeddingManager:
    # model_name is the Hugggingface one
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        # all-MiniLM-L6-v2 : 384 dim
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f'Loading model: {self.model_name}')
            self.model = SentenceTransformer(self.model_name)
            print(f'Loaded model: {self.model.get_sentence_embedding_dimension()} dimensional')
        except Exception as e:
            print(f'Error loading model {self.model_name}, {e}')
            raise

    def generate_embeddings(self, text: List[str]):
        if not self.model:
            raise ValueError('Model not loaded !!!')
        print(f'Generating embeddings for {self.model_name}')
        embeddings = self.model.encode(text)
        print(f'Generated (with shape) {embeddings.shape} dimensional embeddings')
        return embeddings.tolist()

    def get_sentence_embedding_dimension(self):
        if not self.model:
            raise ValueError('Model not loaded !!!')
        return self.model.get_sentence_embedding_dimension()

In [38]:
embedding_manager = EmbeddingManager()

texts = [doc.page_content for doc in split_documents] 

# Normalizing distances
embeddings = embedding_manager.generate_embeddings(texts)
# embeddings = np.array(embeddings)
# embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

Loading model: all-MiniLM-L6-v2
Loaded model: 384 dimensional
Generating embeddings for all-MiniLM-L6-v2
Generated (with shape) (204, 384) dimensional embeddings


# Vector store

In [39]:
class VectorStore:
    def __init__(self, collection_name: str, persist_dir: str='../data/vector_store'):
        self.collection_name = collection_name
        self.persist_dir = persist_dir
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_dir, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_dir)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={'hnsw:space': 'cosine'}
            )
            print(f'VectorStore for {self.collection_name} initialized successfully')
            print(f'Existing numner of docs in the colelction: {self.collection.count()}')
        
        except Exception as e:
            print(f'Error in initializing Vector Store {e}')
            raise 

    def add_docs(self, docs: List[Any], embeddings: np.ndarray):
        if len(docs) != len(embeddings):
            raise ValueError(f'Number of docs ({len(docs)}) must match number of embeddings ({len(embeddings)})...')
        
        print(f'Adding {len(docs)} docs to vectorStore')
    
        ids = []
        metadatas = []
        doc_list = []
        embedding_list = []
        for i, (doc, embedding) in enumerate(zip(docs, embeddings)):
            id = f'doc_{uuid.uuid4().hex[:8]}_{i}'
            ids.append(id)

            doc_list.append(doc.page_content)
            
            embedding_list.append(embedding)
            
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['context_length'] = len(doc.page_content)
            metadatas.append(metadata)

        try:
            print(f'Adding docs and embeddings to collection {self.collection_name}')
            self.collection.add(
                ids=ids,
                embeddings=embedding_list,
                metadatas=metadatas,
                documents=doc_list
            )
            print(f'Successfuly added {len(docs)} documents and embeddings to the collection.')
            print(f'Total docs and embeddings in {self.collection_name} is {self.collection.count()}')
        
        except Exception as e:
            raise ValueError(f'Could not add docs and embeds to collection {self.collection_name}')
        

In [40]:
vectorStore = VectorStore(collection_name='PatchFool_Paper', persist_dir='../data/vector_store/PatchFool')
# vectorStore.add_docs(docs=split_documents, embeddings=embeddings)

VectorStore for PatchFool_Paper initialized successfully
Existing numner of docs in the colelction: 204


# Retriever

In [41]:
class Retriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query: str, k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        # score_threshold: Minimum similarity score threshold
        
        print(f'Retrieving docs for query: {query}')
        print(f'Top k = {k} and score threshold {score_threshold}')

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        # query_embedding = np.array(query_embedding)
        # query_embedding = query_embedding / np.linalg.norm(query_embedding, keepdims=True)
        
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding],
                n_results=k
            )

            retrieved_docs = []

            if results['documents'] and results['metadatas'] and results['distances'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance
                    # print(i, similarity_score, distance)
                    if similarity_score > score_threshold:
                        retrieved_docs.append({
                            'id': id,
                            'document': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i+1
                        })
                print(f'Retrieved {len(retrieved_docs)} after filtering')
            else:
                print(f'No docs found --^^--')
            return retrieved_docs
        except Exception as e:
            raise ValueError(f'Could not retrieve documents : {e}')

In [42]:
retriever = Retriever(vector_store=vectorStore, embedding_manager=embedding_manager)

question = "How better is patchfool compared to its predecessors, give metrics as well?"
retrieved_docs = retriever.retrieve(question)

Retrieving docs for query: How better is patchfool compared to its predecessors, give metrics as well?
Top k = 5 and score threshold 0.0
Generating embeddings for all-MiniLM-L6-v2
Generated (with shape) (1, 384) dimensional embeddings
Retrieved 5 after filtering


In [53]:
retrieved_docs[0]

{'id': 'doc_c71078ff_104',
 'document': 'to the perturbation density, we evaluate our proposed Mild Patch-Fool in Sec. 4.6.\n4.6\nBENCHMARK AGAINST MILD PATCH-FOOL\nSetup. To study the influence of the perturbation strength within each patch, we evaluate our\nproposed Mild Patch-Fool in Sec. 4.6 with L2 or L∞constraints on the patch-wise perturbations with\n8',
 'metadata': {'creationDate': 'D:20250107011300Z',
  'source': '..\\data\\(Minor Project) PatchFool.pdf',
  'keywords': '',
  'source_file': '(Minor Project) PatchFool.pdf',
  'page': 7,
  'trapped': '',
  'author': '',
  'file_path': '..\\data\\(Minor Project) PatchFool.pdf',
  'format': 'PDF 1.5',
  'subject': '',
  'modDate': 'D:20250107011300Z',
  'context_length': 315,
  'total_pages': 18,
  'file_type': 'pdf',
  'doc_index': 104,
  'producer': 'pdfTeX-1.40.25',
  'title': '',
  'moddate': '2025-01-07T01:13:00+00:00',
  'creator': 'LaTeX with hyperref',
  'creationdate': '2025-01-07T01:13:00+00:00'},
 'similarity_score': 0.

# LLM Generation

In [44]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()

True

In [45]:
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

In [46]:
# llm.invoke('Write me a song')

def simple_RAG(query: str, retriever: Retriever, llm: ChatGoogleGenerativeAI, top_k: int = 5):
    results = retriever.retrieve(query=query)
    context = "\n\n".join([i['document'] for i in results])
    if not context:
        return 'No relevant context found.'
    
    template = """
            You are a helpful assistant, for reasearch papers. You haev research papers as data, so you need to be aware of how a research paper is structured, (like where to find the authors of the paper and stuff etc.). Use the following pieces of context to answer the question at the end.
            If you don't know the answer, just say that you don't know, don't try to make up an answer.
            Use five sentences maximum and keep the answer as concise as possible.

            Context: {context}

            Question: {query}

            Helpful Answer:
        """
    
    response = llm.invoke([template.format(context=context, query=query)])
    return response.content

In [47]:
answer = simple_RAG(query=question, retriever=retriever, llm=llm, top_k=5)

Retrieving docs for query: How better is patchfool compared to its predecessors, give metrics as well?
Top k = 5 and score threshold 0.0
Generating embeddings for all-MiniLM-L6-v2
Generated (with shape) (1, 384) dimensional embeddings
Retrieved 5 after filtering


In [48]:
answer

'Based on the provided context, there is no information comparing Patch-Fool to its predecessors. The text discusses "Mild Patch-Fool" and "Sparse Patch-Fool" as variants of Patch-Fool and evaluates different settings or strategies within Patch-Fool itself, such as the number of perturbed patches or L2/L∞ constraints. It also mentions "attention-aware patch selection" as an effective strategy among three for attacking ViTs, but does not identify these as predecessors to Patch-Fool or provide comparative metrics for Patch-Fool against them.'